# 03 — Train a Vision Transformer (ViT-B/16)**Goal:** train a transformer-based architecture on the same task and compare it to the CNN.Vision Transformers split an image into patches and use self-attention to model global relationships. They tend to catch different artifacts than CNNs, which is why combining them in an ensemble works.

## Setup

In [ ]:
import osimport torchimport torch.nn as nnfrom torch.optim import Adamfrom torch.utils.data import DataLoader, Datasetfrom torchvision import transformsfrom torchvision.models import vit_b_16, ViT_B_16_Weightsfrom datasets import load_datasetfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)import numpy as npfrom tqdm import tqdm

## ConfigurationViT is heavier than EfficientNet, so we use a smaller batch size and a much lower learning rate. Transformer fine-tuning is sensitive to LR — `1e-5` is a safe starting point.

In [ ]:
CONFIG = {    "batch_size":      16,    # ViT is heavier than EfficientNet    "epochs":          5,    "lr":              1e-5,  # lower LR for transformer fine-tuning    "weight_decay":    0.01,    "freeze_backbone": True,  # set False for full fine-tuning (needs GPU)    "seed":            42,    "checkpoint":      "checkpoints/vit.pt",}torch.manual_seed(CONFIG["seed"])np.random.seed(CONFIG["seed"])device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Using device: {device}")

## Dataset wrapper (same as in the CNN notebook)

In [ ]:
class DefactifyDataset(Dataset):    def __init__(self, hf_dataset, transform=None):        self.ds = hf_dataset        self.transform = transform    def __len__(self):        return len(self.ds)    def __getitem__(self, idx):        ex = self.ds[idx]        img = ex["Image"].convert("RGB")        if self.transform:            img = self.transform(img)        return (img,                torch.tensor(ex["Label_A"], dtype=torch.long),                torch.tensor(ex["Label_B"], dtype=torch.long))

## Build the ViT modelLoad `vit_b_16` with ImageNet pretraining and replace the head with a fresh 2-class layer.

In [ ]:
def build_vit(num_classes=2, freeze_backbone=True):    """ViT-B/16 with pretrained ImageNet weights and a fresh head."""    model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)    if freeze_backbone:        for param in model.parameters():            param.requires_grad = False    in_features = model.heads.head.in_features    model.heads.head = nn.Sequential(        nn.Dropout(0.3),        nn.Linear(in_features, num_classes),    )    return model

## Load and split data

In [ ]:
print("Loading dataset (filtered: Real, SD3, DALL·E 3)...")raw = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset",                   split="train[:3000]")raw = raw.filter(lambda x: x["Label_B"] in [0, 3, 4])splits   = raw.train_test_split(test_size=0.30, seed=CONFIG["seed"])val_test = splits["test"].train_test_split(test_size=0.50, seed=CONFIG["seed"])train_hf, val_hf, test_hf = splits["train"], val_test["train"], val_test["test"]print(f"Train: {len(train_hf)}  Val: {len(val_hf)}  Test: {len(test_hf)}")preprocess = transforms.Compose([    transforms.Resize((224, 224)),    transforms.ToTensor(),    transforms.Normalize(mean=[0.485, 0.456, 0.406],                         std=[0.229, 0.224, 0.225]),])train_loader = DataLoader(DefactifyDataset(train_hf, preprocess),                          batch_size=CONFIG["batch_size"], shuffle=True)val_loader   = DataLoader(DefactifyDataset(val_hf,   preprocess),                          batch_size=CONFIG["batch_size"])test_loader  = DataLoader(DefactifyDataset(test_hf,  preprocess),                          batch_size=CONFIG["batch_size"])

## Initialize model and optimizer

In [ ]:
model = build_vit(num_classes=2,                  freeze_backbone=CONFIG["freeze_backbone"]).to(device)criterion = nn.CrossEntropyLoss()optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()),                 lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])print("Model ready.")

## Training loop

In [ ]:
def run_epoch(loader, train=True):    model.train() if train else model.eval()    total_loss, correct, total = 0, 0, 0    ctx = torch.enable_grad() if train else torch.no_grad()    with ctx:        for imgs, labels_a, _ in tqdm(loader, desc="Train" if train else "Val"):            imgs, labels_a = imgs.to(device), labels_a.to(device)            if train:                optimizer.zero_grad()            out  = model(imgs)            loss = criterion(out, labels_a)            if train:                loss.backward()                optimizer.step()            total_loss += loss.item()            correct    += (out.argmax(1) == labels_a).sum().item()            total      += labels_a.size(0)    return total_loss / len(loader), correct / totalos.makedirs("checkpoints", exist_ok=True)best_val = 0for epoch in range(CONFIG["epochs"]):    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")    train_loss, train_acc = run_epoch(train_loader, train=True)    val_loss,   val_acc   = run_epoch(val_loader,   train=False)    print(f"  Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}")    if val_acc > best_val:        best_val = val_acc        torch.save(model.state_dict(), CONFIG["checkpoint"])        print(f"  Saved best model (val_acc: {val_acc:.4f})")

## Final test evaluation

In [ ]:
model.load_state_dict(torch.load(CONFIG["checkpoint"]))model.eval()all_preds, all_probs, all_labels, all_label_b = [], [], [], []with torch.no_grad():    for imgs, labels_a, labels_b in tqdm(test_loader, desc="Testing"):        imgs = imgs.to(device)        out  = model(imgs)        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()        preds = out.argmax(1).cpu().numpy()        all_preds.extend(preds)        all_probs.extend(probs)        all_labels.extend(labels_a.numpy())        all_label_b.extend(labels_b.numpy())print("\n" + "=" * 50)print("FINAL TEST METRICS (ViT)")print("=" * 50)print(f"Accuracy:  {accuracy_score(all_labels, all_preds):.4f}")print(f"Precision: {precision_score(all_labels, all_preds):.4f}")print(f"Recall:    {recall_score(all_labels, all_preds):.4f}")print(f"F1 Score:  {f1_score(all_labels, all_preds):.4f}")print(f"ROC-AUC:   {roc_auc_score(all_labels, all_probs):.4f}")# Per-source breakdownall_labels_arr  = np.array(all_labels)all_preds_arr   = np.array(all_preds)all_label_b_arr = np.array(all_label_b)SOURCE_NAMES = {0: "Real", 3: "SD3", 4: "DALL·E 3"}print("\nPer-source accuracy:")for src, name in SOURCE_NAMES.items():    mask = all_label_b_arr == src    if mask.sum() > 0:        acc = (all_preds_arr[mask] == all_labels_arr[mask]).mean()        print(f"  {name:10s}: {acc:.4f} ({mask.sum()} examples)")np.save("checkpoints/vit_test_probs.npy",  np.array(all_probs))np.save("checkpoints/vit_test_labels.npy", np.array(all_labels))print("\nSaved test probabilities for use in 04_ensemble.ipynb.")

## Next stepMove on to **`04_ensemble.ipynb`** to combine the CNN and ViT predictions.